<a href="https://colab.research.google.com/github/sandip01234/Transformer-Model-for-English-to-Nepali-translation/blob/main/transformationtesting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import sentencepiece as spm
from tqdm import tqdm
import math
import os


### file format
Nepali<TAB>    English <br>



In [ ]:
from google.colab import files
uploaded = files.upload()


Saving train.tsv to train (1).tsv


In [ ]:
TRAIN_FILE = "train.tsv" #ename the file name


Train SentencePiece tokenizers

Why?

-Nepali needs subword tokenization

-Better accuracy than word-level

In [ ]:
# def train_tokenizer(input_file, model_prefix, vocab_size=4000):

#     """
#     Train a SentencePiece tokenizer.
#     vocab_size must be <= 4150 for your dataset.
#     """
#     if vocab_size > 4150:
#         print("Warning: vocab_size too high, setting to 4150")
#         vocab_size = 4150

#     spm.SentencePieceTrainer.train(
#         input=input_file,
#         model_prefix=model_prefix,
#         vocab_size=vocab_size,
#         character_coverage=1.0,
#         model_type="unigram",
#         bos_id=1,
#         eos_id=2,
#         unk_id=3,
#         pad_id=0
#     )


import sentencepiece as spm

def train_tokenizer(input_file, model_prefix, max_vocab_size=4000):
    """
    Train a SentencePiece tokenizer.
    If dataset is too small, automatically reduces vocab_size to max allowed.
    """
    # Start with max_vocab_size
    vocab_size = max_vocab_size

    # Try training, reduce vocab_size if error occurs
    while vocab_size > 1000:  # don't go too low
        try:
            spm.SentencePieceTrainer.train(
                input=input_file,
                model_prefix=model_prefix,
                vocab_size=vocab_size,
                character_coverage=1.0,
                model_type="unigram",
                bos_id=1,
                eos_id=2,
                unk_id=3,
                pad_id=0
            )
            print(f"Tokenizer trained with vocab_size={vocab_size}")
            break
        except RuntimeError as e:
            if "Vocabulary size too high" in str(e):
                vocab_size -= 50  # reduce and try again
            else:
                raise e


Split TSV for tokenizer training:

In [ ]:
with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    ne_lines, en_lines = [], []
    for line in f:
        ne, en = line.strip().split("\t")
        ne_lines.append(ne)
        en_lines.append(en)

with open("ne.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(ne_lines))

with open("en.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(en_lines))


Train tokenizers:

In [ ]:
train_tokenizer("ne.txt", "ne_tokenizer")
train_tokenizer("en.txt", "en_tokenizer")

sp_ne = spm.SentencePieceProcessor(model_file="ne_tokenizer.model")
sp_en = spm.SentencePieceProcessor(model_file="en_tokenizer.model")


Tokenizer trained with vocab_size=4000
Tokenizer trained with vocab_size=3750


Input Embedding

In [ ]:
class InputEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.embedding(x) * math.sqrt(self.d_model)


Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000) / d_model))
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)

        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0

        self.d_k = d_model // num_heads
        self.num_heads = num_heads

        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)
        self.fc = nn.Linear(d_model, d_model)

    def forward(self, q, k, v, mask=None):
        B = q.size(0)

        q = self.q(q).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        k = self.k(k).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)
        v = self.v(v).view(B, -1, self.num_heads, self.d_k).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attn = torch.softmax(scores, dim=-1)
        out = (attn @ v)

        out = out.transpose(1, 2).contiguous().view(B, -1, self.num_heads * self.d_k)
        return self.fc(out)


Encoder & Decoder Layers

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, heads, d_ff):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, heads)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x = self.norm1(x + self.attn(x, x, x))
        x = self.norm2(x + self.ff(x))
        return x


Transformer Model

In [ ]:
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=256, heads=4, layers=3):
        super().__init__()
        self.src_embed = InputEmbedding(src_vocab, d_model)
        self.tgt_embed = InputEmbedding(tgt_vocab, d_model)
        self.pos = PositionalEncoding(d_model)

        self.encoder = nn.ModuleList([EncoderLayer(d_model, heads, 1024) for _ in range(layers)])
        self.decoder = nn.ModuleList([EncoderLayer(d_model, heads, 1024) for _ in range(layers)])

        self.fc = nn.Linear(d_model, tgt_vocab)

    def forward(self, src, tgt):
        src = self.pos(self.src_embed(src))
        tgt = self.pos(self.tgt_embed(tgt))

        for layer in self.encoder:
            src = layer(src)

        for layer in self.decoder:
            tgt = layer(tgt)

        return self.fc(tgt)


Dataset & DataLoader

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, file):
        self.data = open(file, encoding="utf-8").read().strip().split("\n")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ne, en = self.data[idx].split("\t")

        src = sp_en.encode(en, out_type=int)
        tgt = sp_ne.encode(ne, out_type=int)

        return torch.tensor(src), torch.tensor(tgt)


Training loop

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Transformer(
    sp_en.vocab_size(),
    sp_ne.vocab_size()
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_fn = nn.CrossEntropyLoss(ignore_index=0)

dataset = TranslationDataset(TRAIN_FILE)
loader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=lambda x: x)

for epoch in range(10):
    model.train()
    total_loss = 0

    for batch in tqdm(loader):
        src, tgt = zip(*batch)

        src = nn.utils.rnn.pad_sequence(src, batch_first=True).to(device)
        tgt = nn.utils.rnn.pad_sequence(tgt, batch_first=True).to(device)

        output = model(src, tgt[:, :-1])
        loss = loss_fn(output.reshape(-1, output.size(-1)), tgt[:, 1:].reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader)}")


100%|██████████| 296/296 [01:36<00:00,  3.06it/s]


Epoch 1, Loss: 6.1875199659450635


100%|██████████| 296/296 [01:34<00:00,  3.14it/s]


Epoch 2, Loss: 5.036438854965004


100%|██████████| 296/296 [01:34<00:00,  3.12it/s]


...

Epoch 8, Loss: 3.0434134658929466


100%|██████████| 296/296 [01:36<00:00,  3.06it/s]


Epoch 9, Loss: 2.829370977910789


100%|██████████| 296/296 [01:35<00:00,  3.09it/s]

Epoch 10, Loss: 2.62594933324569


Inference (Google Translate style

In [ ]:
import torch
from torch.nn.functional import softmax

def translate(sentence, max_len=50, temperature=0.8):  # 1️⃣ add temperature
    model.eval()
    src = torch.tensor(sp_en.encode(sentence)).unsqueeze(0).to(device)
    tgt = [sp_ne.bos_id()]  # BOS token

    for _ in range(max_len):
        tgt_tensor = torch.tensor(tgt).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model(src, tgt_tensor)

        # Take the last token logits
        logits = out[0, -1]  # 2️⃣ get raw logits

        # Apply temperature to soften or sharpen probabilities
        probs = softmax(logits / temperature, dim=-1)  # 3️⃣ divide logits by temperature

        # Sampling instead of argmax to reduce repetition
        next_token = torch.multinomial(probs, 1).item()

        # Optional: prevent repeating same token consecutively
        if len(tgt) > 1 and next_token == tgt[-1]:
            # re-sample if the token is same as last
            top_probs, top_tokens = probs.topk(5)
            next_token = top_tokens[torch.randint(0, 5, (1,))].item()

        # Stop if EOS
        if next_token == sp_ne.eos_id():
            break

        tgt.append(next_token)

    return sp_ne.decode(tgt[1:])  # skip BOS


In [ ]:
print(translate("We should always speak the truth."))


उचाल् फल हुनुहुन्नतादाय्रो थाहा्दा तान बनाउन सक्छौ।िताईन्छ। गहुँितन्छ। जंगलित बिग्र चुक दिनु।ू अघि पेनित भए।ित खसालित टीका उडानित गरियो।ित हुनित भएित चुक कमजोरितरु
